# hillslope site selection

In [ ]:
# Parameters cell
import json
with open('config.json', 'r') as f:
    config = json.load(f)
watershed_name = config['watershed_name']
# hucs           = [config['hucs']]
site_name      = config['site_name']

# get datasets

## get burn severity

In [ ]:
# Load the Libraries
import geopandas as gpd
import pandas as pd
from pathlib import Path
import os
import glob

print("Current working directory:", os.getcwd())
base_path = Path("/global/cfs/cdirs/m1800/xiaoyi/10-Projects/2025-RCSFA-HillslopeFire/MaterialsData/MTBS_fire_WA_2021")

In [ ]:
site_selections_folder = os.path.join (".", "site_selections")
os.makedirs(site_selections_folder, exist_ok=True)

In [ ]:
# =================== read in watershed boundary data =================================
# Read in watershed boundaries using relative paths
ame_boundary = gpd.read_file(base_path / "watershed_boundary_shp_files/AmeMixed_bounds.shp")
#hja_boundary = gpd.read_file(base_path / "watershed_boundary_shp_files/HJAndrewsMixed_bounds.shp")
oak_boundary = gpd.read_file(base_path / "watershed_boundary_shp_files/OakMixed_bounds.shp")

nac_boundary = gpd.read_file(base_path / "watershed_boundary_shp_files/naches.shp")
nac_boundary = nac_boundary.to_crs(ame_boundary.crs)

## verify
print(ame_boundary.crs)
print(nac_boundary.crs)

In [ ]:
# List all shapefiles in the directory
years = ['2021']

for year in years:
    fire_shapefiles = glob.glob(str(base_path / f'MTBS_fire_shp_files/{year}/**/*burn_bndy.shp'), recursive=True)
fire_shapefiles

In [ ]:
# Read all shapefiles and combine into a single GeoDataFrame
mtbs_list = []
for shapefile in fire_shapefiles:
    try:
        gdf = gpd.read_file(shapefile)
        mtbs_list.append(gdf)
    except Exception as e:
        print(f"Error reading {shapefile}: {e}")

# Combine all GeoDataFrames
if mtbs_list:
    mtbs_data = pd.concat(mtbs_list, ignore_index=True)
    mtbs_data = gpd.GeoDataFrame(mtbs_data)
else:
    raise ValueError("No fire shapefiles found or could be read")

# Ensure all geometries have the same CRS
mtbs_data = mtbs_data.to_crs(ame_boundary.crs)


In [ ]:
# Filter MTBS data within each watershed using spatial intersection
ame_fires = gpd.overlay(mtbs_data, ame_boundary, how='intersection')
#hja_fires = gpd.overlay(mtbs_data, hja_boundary, how='intersection')
oak_fires = gpd.overlay(mtbs_data, oak_boundary, how='intersection')
nac_fires = gpd.overlay(mtbs_data, nac_boundary, how='intersection')

# Create output directory if it doesn't exist
output_dir = base_path / "watershed_fire_boundaries"
output_dir.mkdir(exist_ok=True)

# Write the results to shapefiles
ame_fires.to_file(output_dir / "AmeWatershed_Fire_Perimeters_MTBS_2021.shp")
#hja_fires.to_file(output_dir / "HJAndrewsWatershed_Fire_Perimeters_MTBS_1984-2024.shp")
oak_fires.to_file(output_dir / "OakWatershed_Fire_Perimeters_MTBS_2021.shp")
nac_fires.to_file(output_dir / "NachesWatershed_Fire_Perimeters_MTBS_2021.shp")

print("Processing complete! Fire perimeter shapefiles have been created.")

In [ ]:
nac_fires

In [ ]:
import matplotlib.pyplot as plt

# Create a figure with larger size
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# Plot watershed boundaries (just outlines)
ame_boundary.plot(ax=ax, 
                  facecolor='lightblue', 
                  edgecolor='darkblue', 
                  linewidth=2, 
                  alpha=0.3,
                  label='American River Watershed')

oak_boundary.plot(ax=ax, 
                  facecolor='lightgreen', 
                  edgecolor='darkgreen', 
                  linewidth=2, 
                  alpha=0.3,
                  label='Oak Creek Watershed')

nac_boundary.plot(ax=ax, 
                  facecolor='lightgray', 
                  edgecolor='dimgray', 
                  linewidth=2, 
                  alpha=0.3,
                  label='Naches Watershed')

# Plot fire perimeters on top
ame_fires.plot(ax=ax, 
               color='red', 
               alpha=0.8, 
               edgecolor='darkred',
               linewidth=0.5,
               label=f'American River Fires (n={len(ame_fires)})')

oak_fires.plot(ax=ax, 
               color='orange', 
               alpha=0.8, 
               edgecolor='darkorange',
               linewidth=0.5,
               label=f'Oak Creek Fires (n={len(oak_fires)})')

nac_fires.plot(ax=ax, 
               color='red', 
               alpha=0.5, 
               edgecolor='darkred',
               linewidth=0.5,
               label=f'Naches Fires (n={len(oak_fires)})')

# Customize the plot
ax.set_title('Schneider Springs Wildfire Perimeters (2021)', 
             fontsize=16, fontweight='bold')
ax.set_xlabel('Easting', fontsize=12)
ax.set_ylabel('Northing', fontsize=12)

# Add legend
ax.legend(loc='upper right', fontsize=10)

# Add grid
ax.grid(True, alpha=0.3)

# Remove axis spines for cleaner look
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import rasterio
import geopandas as gpd
import matplotlib.pyplot as plt
from rasterio.plot import show
from rasterio.mask import mask
import matplotlib.colors as colors

In [ ]:
selected_path = os.path.join(base_path, "MTBS_fire_shp_files/2021/wa4684012114620210804")
selected_path

In [ ]:
# Paths to the files (update paths based on your directory)
rdnbr_tif_fp = os.path.join(selected_path, "wa4684012114620210804_20210725_20220728_dnbr6.tif")  # Burn severity raster
boundary_shapefile_fp = os.path.join(selected_path, "wa4684012114620210804_20210725_20220728_burn_bndy.shp")  # Fire boundary shapefile

# from src.colormap(1), and turn black to white
# (0, (0, 0, 0, 0)), (1, (0, 100, 0, 255)), (2, (127, 255, 212, 255)), (3, (255, 255, 0, 255)), (4, (255, 0, 0, 255)), (5, (127, 255, 0, 255))
# Legend for burn severity values (you may need to verify the mapping from metadata)
burn_severity_colors = {
    0: (1,1,1),                # "Non-Processing Area" - white
    1: (0,100/255,0),          # "Unburned to Low"     - green
    2: (127/255,1,212/255),    # "Low" Severity        - cyan
    3: (1,1,0),                # "Moderate" Severity   - yellow
    4: (1,0,0),                # "High" Severity       - red
    5: (127/255,1,0),          # "Increased Greenness" - light green
}

# Load the burn severity raster
with rasterio.open(rdnbr_tif_fp) as src:
    burn_severity_data = src.read(1)  # Read first band
    profile = src.profile
    raw_colormap = src.colormap(1)
    # cmap = colors.ListedColormap(
    #         [(val[0]/255.0, val[1]/255.0, val[2]/255.0) for key, val in raw_colormap.items()]  # Map pixel values to RGB colors
    #     )

# # Load the fire boundary shapefile
# boundary = gpd.read_file(boundary_shapefile_fp)

# Create a custom colormap for burn severity
cmap = colors.ListedColormap([burn_severity_colors[key] for key in burn_severity_colors])
bounds = [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5, 5.5]  # Boundaries between categories
norm = colors.BoundaryNorm(boundaries=bounds, ncolors=6)

# Plot the burn severity with boundary outline overlay
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
show(burn_severity_data, ax=ax, cmap=cmap)
#boundary.plot(ax=ax, facecolor="none", edgecolor="yellow", linewidth=1)

# Add color bar for burn severity
cbar = plt.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax)
tick_positions = [0, 1, 2, 3, 4, 5]  # Center of each category
cbar.set_ticks(tick_positions)
cbar.ax.set_yticklabels([
    "Non-Processing Area", 
    "Unburned to Low", 
    "Low", 
    "Moderate",
    "High", 
    "Increased Greenness"
])
plt.title("Burn Severity Map with Fire Boundary")
plt.axis('off')

plt.show()

In [ ]:
burn_severity_data.shape

In [ ]:
burn_severity_data.min()

In [ ]:
burn_severity_data.max()

In [ ]:
profile

In [ ]:
import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from rasterio.warp import transform_bounds

# Define burn severity colormap
burn_severity_colors = {
    0: (1, 1, 1),              # "Non-Processing Area" - white
    1: (0, 100/255, 0),        # "Unburned to Low"     - green
    2: (127/255, 1, 212/255),  # "Low" Severity        - cyan
    3: (1, 1, 0),              # "Moderate" Severity   - yellow
    4: (1, 0, 0),              # "High" Severity       - red
    5: (127/255, 1, 0),        # "Increased Greenness" - light green
}

cmap = colors.ListedColormap([burn_severity_colors[key] for key in burn_severity_colors])
bounds = [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5, 5.5]
norm = colors.BoundaryNorm(boundaries=bounds, ncolors=6)

# Create figure
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# Load burn severity raster and get its CRS
with rasterio.open(rdnbr_tif_fp) as src:
    burn_severity_data = src.read(1)
    raster_crs = src.crs
    raster_transform = src.transform
    raster_bounds = src.bounds
    
    print(f"Raster CRS: {raster_crs}")
    print(f"Raster bounds: {raster_bounds}")
    
    # Check if we need to reproject the vector data
    vector_crs = nac_boundary.crs
    print(f"Vector CRS: {vector_crs}")
    
    # Reproject vector data to match raster CRS if needed
    print("Reprojecting vector data to match raster CRS...")
    ame_boundary_proj = ame_boundary.to_crs(raster_crs)
    oak_boundary_proj = oak_boundary.to_crs(raster_crs)
    nac_boundary_proj = nac_boundary.to_crs(raster_crs)
    nac_fires_proj = nac_fires.to_crs(raster_crs)
    
    # Plot raster using show() which handles the transform properly
    show(burn_severity_data, transform=raster_transform, ax=ax, cmap=cmap, norm=norm, zorder=1)

# Plot watershed boundaries
ame_boundary_proj.plot(ax=ax, 
                       facecolor='none', 
                       edgecolor='darkblue', 
                       linewidth=1.5, 
                       linestyle='-',
                       label='American River Watershed',
                       zorder=2)

oak_boundary_proj.plot(ax=ax, 
                       facecolor='none', 
                       edgecolor='darkgreen', 
                       linewidth=1.5, 
                       linestyle='-',
                       label='Oak Creek Watershed',
                       zorder=2)

nac_boundary_proj.plot(ax=ax, 
                       facecolor='none', 
                       edgecolor='black', 
                       linewidth=2, 
                       linestyle='-',
                       label='Naches Watershed',
                       zorder=2)

# # Plot fire perimeters
# nac_fires_proj.plot(ax=ax, 
#                     facecolor='none',
#                     edgecolor='magenta', 
#                     linewidth=2,
#                     linestyle='-.',
#                     label='Naches Fire Perimeter',
#                     zorder=3)

# Add colorbar for burn severity
cbar = plt.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax, fraction=0.046, pad=0.04)
tick_positions = [0, 1, 2, 3, 4, 5]
cbar.set_ticks(tick_positions)
cbar.ax.set_yticklabels([
    "Non-Processing Area", 
    "Unburned to Low", 
    "Low", 
    "Moderate",
    "High", 
    "Increased Greenness"
], fontsize=10)
cbar.set_label('Burn Severity', fontsize=12, fontweight='bold')

# Customize plot
ax.set_title('Schneider Springs Fire Burn Severity (2021/8/4)', 
             fontsize=16, fontweight='bold')
ax.set_xlabel('Easting', fontsize=12)
ax.set_ylabel('Northing', fontsize=12)

# Add legend
ax.legend(loc='upper right', fontsize=11, framealpha=0.9)

# Add grid
ax.grid(True, alpha=0.3, zorder=0)

plt.tight_layout()
plt.show()

## get stream network

In [ ]:
# Essential imports for watershed workflow
import watershed_workflow
import watershed_workflow.source_list
import watershed_workflow.plot
import pyproj

# Set up watershed workflow CRS (DayMet CRS)
crs_daymet = watershed_workflow.crs.daymet_crs()

# Set up sources
sources = watershed_workflow.source_list.get_default_sources()
sources['hydrography'] = watershed_workflow.source_list.hydrography_sources['NHD Plus']
sources['HUC'] = watershed_workflow.source_list.huc_sources['NHD Plus']

# Parameters for river extraction
hucs_config = [config['hucs']]
ignore_small_rivers = 2
prune_by_area_fraction = 0.0

# Get HUC12 list
def get_huc12(hucs):
    huc12_list = []
    for huc in hucs:
        if len(huc) == 12:
            huc12_list.append(huc)
        elif len(huc) == 10:
            for i in range(1,20):
                huc12_list.append(huc+str(i).zfill(2))
        elif len(huc) == 8:
            for i in range(1,20):
                for j in range(1,20):
                    huc12_list.append(huc+str(i).zfill(2)+str(j).zfill(2))
    return huc12_list

hucs = get_huc12(hucs_config)
huc_level = 12

print(f"Processing HUCs: {hucs[:5]}...")

# Load watershed HUCs
my_hucs = []
for huc in hucs:
    _, ws = watershed_workflow.get_hucs(sources['HUC'], huc, huc_level, crs_daymet)
    my_hucs.extend(ws)

watershed = watershed_workflow.split_hucs.SplitHUCs(my_hucs)

# Download/collect the river network
print("Downloading river network...")
_, reaches = watershed_workflow.get_reaches(sources['hydrography'], hucs[0], 
                                            watershed.exterior(), crs_daymet, raster_crs,
                                            in_network=True, properties=True)

# Construct river network
rivers = watershed_workflow.construct_rivers(reaches, method='hydroseq',
                                             ignore_small_rivers=ignore_small_rivers,
                                             prune_by_area=prune_by_area_fraction * watershed.exterior().area * 1.e-6,
                                             remove_diversions=True,
                                             remove_braided_divergences=True)

print(f"Number of rivers: {len(rivers)}")

In [ ]:
# Set up pyproj transformer from DayMet CRS to raster CRS
# transformer = pyproj.Transformer.from_crs(
#     crs_daymet.to_string(),
#     raster_crs.to_string(),
#     always_xy=True
# )

# Reproject rivers to raster CRS using pyproj
import shapely.geometry as sg

# rivers_reprojected = []
# for river in rivers:
#     all_coords = []
#     for node in river.preOrder():
#         coords = list(node.segment.coords)
#         for x, y in coords:
#             x_new, y_new = transformer.transform(x, y)
#             all_coords.append((x_new, y_new))
    
#     if len(all_coords) > 1:
#         rivers_reprojected.append(sg.LineString(all_coords))

# print(f"Reprojected {len(rivers_reprojected)} river segments")

# Plot burn severity with reprojected rivers
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# Plot burn severity raster
show(burn_severity_data, transform=raster_transform, ax=ax, cmap=cmap, norm=norm, zorder=1)

# Plot reprojected rivers
# for river_line in rivers_reprojected:
#     x, y = river_line.xy
#     ax.plot(x, y, color='cyan', linewidth=1.5, alpha=0.8, zorder=4)
watershed_workflow.plot.rivers(rivers, raster_crs, ax=ax, colors='b', linewidth=0.5)

# Plot watershed boundaries (already reprojected)
nac_boundary_proj.boundary.plot(ax=ax, color='black', linewidth=2, 
                                 label='Naches Watershed', zorder=2)

# Add colorbar for burn severity
cbar = plt.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax, fraction=0.046, pad=0.04)
tick_positions = [0, 1, 2, 3, 4, 5]
cbar.set_ticks(tick_positions)
cbar.ax.set_yticklabels([
    "Non-Processing Area", 
    "Unburned to Low", 
    "Low", 
    "Moderate",
    "High", 
    "Increased Greenness"
], fontsize=10)
cbar.set_label('Burn Severity', fontsize=12, fontweight='bold')

# Customize plot
ax.set_title('Schneider Springs Fire Burn Severity with River Network', 
             fontsize=16, fontweight='bold')
ax.set_xlabel('Easting', fontsize=12)
ax.set_ylabel('Northing', fontsize=12)

# Add legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='black', linewidth=2, label='Naches Watershed'),
    Line2D([0], [0], color='cyan', linewidth=1.5, label='River Network')
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=11, framealpha=0.9)

# Add grid
ax.grid(True, alpha=0.3, zorder=0)

plt.tight_layout()
plt.show()

## get DEM

In [ ]:
sources = watershed_workflow.source_list.get_default_sources()
sources['DEM'] = watershed_workflow.source_list.dem_sources['NED 1/3 arc-second']

In [ ]:
# download the needed rasters
dem_profile2, dem2 = watershed_workflow.get_raster_on_shape(sources['DEM'], watershed.exterior(), crs_daymet, out_crs=raster_crs)

# Calculate extent from the transform for geographic coordinates
transform = dem_profile2['transform']
rows, cols = dem2.shape
left = transform.c
right = transform.c + cols * transform.a
top = transform.f
bottom = transform.f + rows * transform.e
extent = [left, right, bottom, top]

fig, ax = plt.subplots(1,1, figsize=(10,10))
im1 = ax.imshow(dem2, cmap='terrain', extent=extent, origin='upper')
#ax.set_title('DEM under Daymet CRS')
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
fig.colorbar(im1, ax=ax, orientation='horizontal', pad=0.1, label='Elevation (Z)')

nac_boundary_proj.boundary.plot(ax=ax, color='black', linewidth=2, 
                                 label='Naches Watershed', zorder=2)
watershed_workflow.plot.rivers(rivers, raster_crs, ax=ax, color='red', linewidth=0.5)

## get MODIS-LAI prefire and postfire

In [ ]:
## refer ./get_MODIS-LAI.ipynb

In [ ]:
# Essential imports (if not already loaded)
import xarray as xr
from shapely.geometry import mapping
import numpy as np

# Load MODIS LAI data
data_raw_dir = f'./MODIS_raw/{watershed_name}/{watershed_name}-square'
fname_lai = data_raw_dir + '/MCD15A3H.061_500m_aid0001.nc'

# Open and process LAI dataset
dset = xr.open_dataset(fname_lai)
data = dset.Lai_500m

# Mask data (remove extreme values)
mask_data = data.where((data < 6.5) & (data >= 0))

# Clip to Naches watershed
mask_data.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
mask_data.rio.write_crs("epsg:4326", inplace=True)
clipped_data = mask_data.rio.clip(nac_boundary.geometry.apply(mapping), nac_boundary.crs, drop=True)

LAI_data = clipped_data

t_lai_prefire  = int(1751)
t_lai_postfire = int(1760)

# Schneider Springs Fire: 2021-8-4 evening
# Pre-fire time index: 1751, Post-fire time index: 1763
print("Pre-fire time: " + LAI_data.time[t_lai_prefire].dt.strftime('%Y-%m-%d').item())
print("Post-fire time: " + LAI_data.time[t_lai_postfire].dt.strftime('%Y-%m-%d').item())

# Plot pre-fire vs post-fire LAI comparison
fig, axs = plt.subplots(1, 3, figsize=(16, 3))
ax1, ax2, ax3 = axs[0], axs[1], axs[2]

# Pre-fire LAI
LAI_data.isel(time=t_lai_prefire).plot(ax=ax1, levels=np.linspace(0, 5, 51), cmap='Spectral_r', extend='max')
ax1.set_title(f"Pre-fire LAI\n{LAI_data.time[t_lai_prefire].dt.strftime('%Y-%m-%d').item()}")
nac_boundary.to_crs("epsg:4326").boundary.plot(ax=ax1, color='black', linewidth=1.5)
ax1.set_aspect('equal')

# Post-fire LAI
LAI_data.isel(time=t_lai_postfire).plot(ax=ax2, levels=np.linspace(0, 5, 51), cmap='Spectral_r', extend='max')
ax2.set_title(f"Post-fire LAI\n{LAI_data.time[t_lai_postfire].dt.strftime('%Y-%m-%d').item()}")
nac_boundary.to_crs("epsg:4326").boundary.plot(ax=ax2, color='black', linewidth=1.5)
ax2.set_aspect('equal')

# Difference (pre - post)
dif = LAI_data.isel(time=t_lai_prefire) - LAI_data.isel(time=t_lai_postfire)
dif.plot(ax=ax3, levels=np.linspace(-2.5, 2.5, 51), cmap='coolwarm', extend='both')
ax3.set_title('LAI Difference\n(Pre-fire - Post-fire)')
nac_boundary.to_crs("epsg:4326").boundary.plot(ax=ax3, color='black', linewidth=1.5)
ax3.set_aspect('equal')

plt.tight_layout()
plt.show()

In [ ]:
print(data.shape)
print(clipped_data.shape)

In [ ]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.transform import from_bounds
import numpy as np

# Calculate LAI difference
lai_prefire = LAI_data.isel(time=t_lai_prefire)
lai_postfire = LAI_data.isel(time=t_lai_postfire)
lai_diff = lai_prefire - lai_postfire

# Get the burn severity raster CRS and bounds
with rasterio.open(rdnbr_tif_fp) as src:
    burn_crs = src.crs
    burn_transform = src.transform
    burn_bounds = src.bounds
    burn_severity_data = src.read(1)
    
print(f"Burn severity CRS: {burn_crs}")
print(f"LAI data CRS: EPSG:4326")

# Reproject LAI difference to burn severity CRS
lai_diff_values = lai_diff.values
lai_lons = lai_diff.lon.values
lai_lats = lai_diff.lat.values

# Create transform for LAI data
lai_transform = from_bounds(
    lai_lons.min(), lai_lats.min(), 
    lai_lons.max(), lai_lats.max(),
    len(lai_lons), len(lai_lats)
)

# Calculate the transform for reprojected data
dst_transform, dst_width, dst_height = calculate_default_transform(
    'EPSG:4326',  # source CRS
    burn_crs,      # destination CRS
    len(lai_lons), len(lai_lats),
    lai_lons.min(), lai_lats.min(),
    lai_lons.max(), lai_lats.max()
)

# Create destination array
lai_diff_reprojected = np.empty((dst_height, dst_width), dtype=np.float32)

# Perform reprojection
reproject(
    source=lai_diff_values,
    destination=lai_diff_reprojected,
    src_transform=lai_transform,
    src_crs='EPSG:4326',
    dst_transform=dst_transform,
    dst_crs=burn_crs,
    resampling=Resampling.bilinear
)

# Reproject boundaries
nac_boundary_burn_crs = nac_boundary.to_crs(burn_crs)
nac_fires_burn_crs = nac_fires.to_crs(burn_crs)

# Create two-panel plot
fig, axs = plt.subplots(1, 2, figsize=(16, 6))
ax1, ax2 = axs[0], axs[1]

# Panel 1: LAI difference with Naches boundary and fire boundary
extent1 = [dst_transform.c, dst_transform.c + dst_transform.a * dst_width,
           dst_transform.f + dst_transform.e * dst_height, dst_transform.f]
im1 = ax1.imshow(lai_diff_reprojected, cmap='coolwarm', 
                 extent=extent1, vmin=-2.5, vmax=2.5, zorder=1)
nac_boundary_burn_crs.boundary.plot(ax=ax1, color='black', linewidth=2, 
                                     label='Naches Watershed', zorder=2)
nac_fires_burn_crs.boundary.plot(ax=ax1, color='magenta', linewidth=2, 
                                  linestyle='--', label='Fire Perimeter', zorder=3)
ax1.set_title('LAI Difference (Pre-fire - Post-fire)\nwith Fire Boundary', fontsize=14, fontweight='bold')
ax1.set_xlabel('Easting', fontsize=11)
ax1.set_ylabel('Northing', fontsize=11)
ax1.legend(loc='upper right', fontsize=10, framealpha=0.9)
ax1.grid(True, alpha=0.3)
cbar1 = plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
cbar1.set_label('LAI Difference [-]', fontsize=11)

# Panel 2: LAI difference with Naches boundary and burn severity (masked)
# Create masked burn severity (set 0 values to NaN for transparency)
burn_severity_masked = np.where(burn_severity_data == 0, np.nan, burn_severity_data)

# Define colormap without the "Non-Processing Area"
burn_severity_colors_active = {
    1: (0, 100/255, 0),        # "Unburned to Low"     - green
    2: (127/255, 1, 212/255),  # "Low" Severity        - cyan
    3: (1, 1, 0),              # "Moderate" Severity   - yellow
    4: (1, 0, 0),              # "High" Severity       - red
    5: (127/255, 1, 0),        # "Increased Greenness" - light green
}
cmap_burn = colors.ListedColormap([burn_severity_colors_active[key] for key in burn_severity_colors_active])
bounds_burn = [0.5, 1.5, 2.5, 3.5, 4.5, 5.5]
norm_burn = colors.BoundaryNorm(boundaries=bounds_burn, ncolors=5)

# Plot burn severity (masked)
show(burn_severity_masked, transform=burn_transform, ax=ax2, 
     cmap=cmap_burn, norm=norm_burn, zorder=2, alpha=1)

# Overlay LAI difference with transparency
im2 = ax2.imshow(lai_diff_reprojected, cmap='coolwarm', 
                 extent=extent1, vmin=-2.5, vmax=2.5, alpha=1, zorder=1)

# Plot boundaries
nac_boundary_burn_crs.boundary.plot(ax=ax2, color='black', linewidth=2, 
                                     label='Naches Watershed', zorder=3)

# Add colorbar for burn severity
cbar_burn = plt.colorbar(plt.cm.ScalarMappable(cmap=cmap_burn, norm=norm_burn), 
                         ax=ax2, fraction=0.046, pad=0.1, location='left')
tick_positions = [1, 2, 3, 4, 5]
cbar_burn.set_ticks(tick_positions)
cbar_burn.ax.set_yticklabels([
    "Unburned to Low", 
    "Low", 
    "Moderate",
    "High", 
    "Increased Greenness"
], fontsize=9)
cbar_burn.set_label('Burn Severity', fontsize=10, fontweight='bold')

ax2.set_title('LAI Difference with Burn Severity', fontsize=14, fontweight='bold')
ax2.set_xlabel('Easting', fontsize=11)
ax2.set_ylabel('Northing', fontsize=11)
ax2.legend(loc='upper right', fontsize=10, framealpha=0.9)
ax2.grid(True, alpha=0.3)
cbar2 = plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
cbar2.set_label('LAI Difference [-]', fontsize=11)

plt.tight_layout()
plt.show()

print(f"Reprojected LAI difference shape: {lai_diff_reprojected.shape}")
print(f"Burn severity shape: {burn_severity_data.shape}")

In [ ]:
# For each burn severity raster pixel (within the mask), obtain its LAI difference value

# Get valid burn severity pixels (non-zero values)
valid_mask = burn_severity_data != 0

# Get the pixel coordinates where burn severity is valid
valid_rows, valid_cols = np.where(valid_mask)

print(f"Total burn severity pixels (non-zero): {len(valid_rows)}")

# For each valid pixel, get its coordinates and corresponding LAI difference value
burn_lai_data = []

for row, col in zip(valid_rows, valid_cols):
    # Get burn severity value
    burn_severity_val = burn_severity_data[row, col]
    
    # Get the geographic coordinates of this pixel in burn severity CRS
    x_coord_geo, y_coord_geo = rasterio.transform.xy(burn_transform, row, col, offset='center')
    
    # Convert geographic coordinates to pixel indices in LAI difference raster
    # Use the inverse of dst_transform to go from geographic coords to pixel indices
    lai_col, lai_row = ~dst_transform * (x_coord_geo, y_coord_geo)
    lai_row_int = int(round(lai_row))
    lai_col_int = int(round(lai_col))
    
    # Check if this coordinate is within the reprojected LAI difference bounds
    if (0 <= lai_row_int < lai_diff_reprojected.shape[0] and 
        0 <= lai_col_int < lai_diff_reprojected.shape[1]):
        lai_diff_val = lai_diff_reprojected[lai_row_int, lai_col_int]
    else:
        lai_diff_val = np.nan
    
    burn_lai_data.append({
        'burn_row': row,
        'burn_col': col,
        'lai_row': lai_row_int if not np.isnan(lai_diff_val) else np.nan,
        'lai_col': lai_col_int if not np.isnan(lai_diff_val) else np.nan,
        'x_geo': x_coord_geo,
        'y_geo': y_coord_geo,
        'burn_severity': burn_severity_val,
        'lai_difference': lai_diff_val
    })

# Convert to DataFrame for easier analysis
burn_lai_df = pd.DataFrame(burn_lai_data)

# Remove rows with NaN LAI difference values
burn_lai_df_clean = burn_lai_df.dropna(subset=['lai_difference'])

print(f"Valid pixels with both burn severity and LAI difference: {len(burn_lai_df_clean)}")
print(f"Pixels outside LAI coverage: {len(burn_lai_df) - len(burn_lai_df_clean)}")
print("\nFirst few rows:")
print(burn_lai_df_clean.head(10))

# Summary statistics by burn severity class
print("\n" + "="*60)
print("LAI Difference Statistics by Burn Severity Class:")
print("="*60)
severity_labels = {
    1: "Unburned to Low",
    2: "Low Severity",
    3: "Moderate Severity",
    4: "High Severity",
    5: "Increased Greenness"
}

for severity_class in sorted(burn_lai_df_clean['burn_severity'].unique()):
    subset = burn_lai_df_clean[burn_lai_df_clean['burn_severity'] == severity_class]
    label = severity_labels.get(severity_class, f"Class {severity_class}")
    print(f"\n{label} (Class {severity_class}):")
    print(f"  Count: {len(subset)}")
    print(f"  LAI Difference - Mean: {subset['lai_difference'].mean():.3f}")
    print(f"  LAI Difference - Std:  {subset['lai_difference'].std():.3f}")
    print(f"  LAI Difference - Min:  {subset['lai_difference'].min():.3f}")
    print(f"  LAI Difference - Max:  {subset['lai_difference'].max():.3f}")

# # Create a scatter plot: Burn Severity vs LAI Difference
# fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# for severity_class in sorted(burn_lai_df_clean['burn_severity'].unique()):
#     subset = burn_lai_df_clean[burn_lai_df_clean['burn_severity'] == severity_class]
#     label = severity_labels.get(severity_class, f"Class {severity_class}")
#     color = burn_severity_colors_active[severity_class]
#     ax.scatter(subset['burn_severity'], subset['lai_difference'], 
#                alpha=0.3, s=10, color=color, label=label)

# ax.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
# ax.set_xlabel('Burn Severity Class', fontsize=12)
# ax.set_ylabel('LAI Difference (Pre-fire - Post-fire)', fontsize=12)
# ax.set_title('LAI Difference vs Burn Severity\n(Positive = vegetation loss, Negative = vegetation increase)', 
#              fontsize=14, fontweight='bold')
# ax.legend(fontsize=10)
# ax.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()

# Box plot by burn severity class
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

data_by_class = [burn_lai_df_clean[burn_lai_df_clean['burn_severity'] == sc]['lai_difference'].values 
                 for sc in sorted(burn_lai_df_clean['burn_severity'].unique())]
labels_list = [severity_labels.get(sc, f"Class {sc}") 
               for sc in sorted(burn_lai_df_clean['burn_severity'].unique())]

bp = ax.boxplot(data_by_class, labels=labels_list, patch_artist=True,
                showmeans=False,  # Show mean as a marker
                meanline=True,  # Show mean as a line
                medianprops=dict(color='darkblue', linewidth=2),  # Median line color
                flierprops=dict(marker='o', markerfacecolor='gray', markersize=4, alpha=0.5),  # Outlier style
                whiskerprops=dict(linewidth=1.5),
                capprops=dict(linewidth=1.5))

# Color the boxes according to burn severity
for patch, severity_class in zip(bp['boxes'], sorted(burn_lai_df_clean['burn_severity'].unique())):
    patch.set_facecolor(burn_severity_colors_active[severity_class])
    patch.set_alpha(0.6)

ax.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.set_ylabel('LAI Difference (Pre-fire - Post-fire)', fontsize=12)
ax.set_title('Distribution of LAI Difference by Burn Severity Class', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=15, ha='right')

plt.tight_layout()
plt.show()

## (to-do) get DOC injection prefire and postfire

In [ ]:
## refer ./ELM_from_huilin/get_docflux_from_ELM_3D.ipynb

# Approach 1 - 1km burn severity grid + LAI difference

## Select the grid ~900m res

In [ ]:
## raw burn severity map is 30m resolution

In [ ]:
import rasterio

# Load the burn severity raster
with rasterio.open(rdnbr_tif_fp) as src:
    profile = src.profile
    transform = src.transform
    
    # Get pixel resolution
    pixel_width = transform.a   # or transform[0]
    pixel_height = -transform.e  # or -transform[4] (negative because usually counts down)
    
    print(f"Pixel width (x-resolution): {pixel_width} meters")
    print(f"Pixel height (y-resolution): {pixel_height} meters")
    print(f"\nFull transform: {transform}")
    print(f"\nRaster dimensions: {src.width} x {src.height} pixels")
    print(f"CRS: {src.crs}")

In [ ]:
## downscale burn severity raster to 900m

In [ ]:
import numpy as np
from rasterio.windows import Window
import rasterio
from rasterio.transform import Affine

# Define weights for each burn severity class
severity_weights = {
    0: 0,   # "Non-Processing Area" - exclude
    1: 0,   # "Unburned to Low" - no impact
    2: 1,   # "Low" Severity
    3: 2,   # "Moderate" Severity
    4: 3,   # "High" Severity
    5: -1,  # "Increased Greenness" - negative impact
}

# Load the 30m burn severity data
with rasterio.open(rdnbr_tif_fp) as src:
    burn_severity_30m = src.read(1)
    transform_30m = src.transform
    crs = src.crs
    
    print(f"Original 30m resolution: {src.res}")
    print(f"Original shape: {burn_severity_30m.shape}")
    
    # Calculate new dimensions for 900m resolution (30 pixels = 900m)
    aggregation_factor = 30  # 30 pixels of 30m = 900m
    
    new_height = burn_severity_30m.shape[0] // aggregation_factor
    new_width = burn_severity_30m.shape[1] // aggregation_factor
    
    print(f"\nNew 900m shape: ({new_height}, {new_width})")
    
    # Initialize the 900m burn severity array
    burn_severity_900m = np.zeros((new_height, new_width), dtype=np.float32)
    
    # Aggregate pixels with weighted average
    for i in range(new_height):
        for j in range(new_width):
            # Get the 30x30 window of 30m pixels
            row_start = i * aggregation_factor
            row_end = (i + 1) * aggregation_factor
            col_start = j * aggregation_factor
            col_end = (j + 1) * aggregation_factor
            
            window_data = burn_severity_30m[row_start:row_end, col_start:col_end]
            
            # Calculate weighted average
            total_weighted_sum = 0
            total_weight = 0
            
            for severity_class, weight in severity_weights.items():
                # Count pixels of this severity class
                pixel_count = np.sum(window_data == severity_class)
                
                # Add to weighted sum (ignore class 0 with weight 0)
                if weight != 0 or severity_class != 0:
                    total_weighted_sum += pixel_count * weight
                    if severity_class != 0:  # Don't count non-processing pixels in total
                        total_weight += pixel_count
            
            # Calculate weighted average severity
            if total_weight > 0:
                burn_severity_900m[i, j] = total_weighted_sum / total_weight
            else:
                burn_severity_900m[i, j] = np.nan  # No valid data in this cell
        
        # Progress indicator
        if (i + 1) % 10 == 0:
            print(f"Processing row {i+1}/{new_height}...")
    
    # Create new transform for 900m resolution
    transform_900m = Affine(
        transform_30m.a * aggregation_factor,  # x pixel size
        transform_30m.b,
        transform_30m.c,  # x origin
        transform_30m.d,
        transform_30m.e * aggregation_factor,  # y pixel size
        transform_30m.f   # y origin
    )
    
    print(f"\nNew 900m resolution: {transform_900m.a} x {-transform_900m.e} meters")
    print(f"Weighted average burn severity range: [{np.nanmin(burn_severity_900m):.2f}, {np.nanmax(burn_severity_900m):.2f}]")
    
# Plot comparison: original 30m vs aggregated 900m
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot original 30m burn severity
ax1 = axes[0]
im1 = ax1.imshow(burn_severity_30m, cmap=cmap, norm=norm)
ax1.set_title('Original 30m Burn Severity\n(Categorical)', fontsize=14, fontweight='bold')
ax1.axis('off')
cbar1 = plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
cbar1.set_label('Severity Class', fontsize=11)

# Plot aggregated 900m weighted burn severity
ax2 = axes[1]
im2 = ax2.imshow(burn_severity_900m, cmap='YlOrRd', vmin=0, vmax=3)
ax2.set_title('Aggregated 900m Burn Severity\n(Weighted Average)', fontsize=14, fontweight='bold')
ax2.axis('off')
cbar2 = plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
cbar2.set_label('Weighted Severity', fontsize=11)

plt.tight_layout()
plt.show()

# Save the 900m burn severity raster
output_900m_path = os.path.join(site_selections_folder, "burn_severity_900m_weighted.tif")
with rasterio.open(
    output_900m_path,
    'w',
    driver='GTiff',
    height=new_height,
    width=new_width,
    count=1,
    dtype=burn_severity_900m.dtype,
    crs=crs,
    transform=transform_900m,
    nodata=np.nan
) as dst:
    dst.write(burn_severity_900m, 1)

print(f"\nSaved 900m weighted burn severity raster to:\n{output_900m_path}")

In [ ]:
## resolution of LAI data

In [ ]:
# Method 1: Check the coordinate spacing
lon_spacing = np.abs(LAI_data.lon.values[1] - LAI_data.lon.values[0])
lat_spacing = np.abs(LAI_data.lat.values[1] - LAI_data.lat.values[0])

print(f"Longitude spacing: {lon_spacing:.6f} degrees")
print(f"Latitude spacing: {lat_spacing:.6f} degrees")

# Convert to approximate meters (at mid-latitude)
# At ~46°N latitude (approximate for Naches watershed)
mid_lat = np.mean(LAI_data.lat.values)
lon_spacing_m = lon_spacing * 111320 * np.cos(np.radians(mid_lat))
lat_spacing_m = lat_spacing * 111320

print(f"\nApproximate resolution:")
print(f"  Longitude: {lon_spacing_m:.1f} meters")
print(f"  Latitude: {lat_spacing_m:.1f} meters")

# Method 2: Check the rio accessor if available
try:
    resolution = LAI_data.rio.resolution()
    print(f"\nRio resolution: {resolution}")
except:
    print("\nRio resolution not available")

In [ ]:
# Method 1: Resample LAI difference to 900m resolution
# This aggregates the higher resolution LAI data to match your 900m burn severity grid

# First, load the burn severity raster info for 900m grid
with rasterio.open(output_900m_path) as src_900m:
    transform_900m = src_900m.transform
    profile_900m = src_900m.profile
    burn_900m_shape = (src_900m.height, src_900m.width)
    burn_900m_crs = src_900m.crs

print(f"900m grid shape: {burn_900m_shape}")
print(f"900m grid transform: {transform_900m}")

# Reproject LAI difference to 900m resolution using the same CRS and transform as burn severity
from rasterio.warp import reproject, Resampling

# Calculate LAI difference (if not already done)
lai_prefire = LAI_data.isel(time=t_lai_prefire)
lai_postfire = LAI_data.isel(time=t_lai_postfire)
lai_diff = lai_prefire - lai_postfire

lai_diff_values = lai_diff.values
lai_lons = lai_diff.lon.values
lai_lats = lai_diff.lat.values

# Create transform for LAI data
lai_transform = from_bounds(
    lai_lons.min(), lai_lats.min(), 
    lai_lons.max(), lai_lats.max(),
    len(lai_lons), len(lai_lats)
)

# Create destination array for 900m LAI difference
lai_diff_900m = np.empty(burn_900m_shape, dtype=np.float32)

# Reproject LAI difference directly to 900m grid
reproject(
    source=lai_diff_values,
    destination=lai_diff_900m,
    src_transform=lai_transform,
    src_crs='EPSG:4326',
    dst_transform=transform_900m,
    dst_crs=burn_900m_crs,
    resampling=Resampling.average  # Use average to aggregate multiple LAI pixels
)

print(f"LAI difference 900m shape: {lai_diff_900m.shape}")
print(f"LAI difference 900m range: [{np.nanmin(lai_diff_900m):.3f}, {np.nanmax(lai_diff_900m):.3f}]")

# Create a dataframe combining burn severity and LAI difference at 900m
data_900m = []
for i in range(burn_900m_shape[0]):
    for j in range(burn_900m_shape[1]):
        burn_val = burn_severity_900m[i, j]
        lai_val = lai_diff_900m[i, j]
        
        # Get geographic coordinates of cell center
        x_geo, y_geo = rasterio.transform.xy(transform_900m, i, j, offset='center')
        
        if not np.isnan(burn_val) and not np.isnan(lai_val):
            data_900m.append({
                'row': i,
                'col': j,
                'x_geo': x_geo,
                'y_geo': y_geo,
                'burn_severity_weighted': burn_val,
                'lai_difference': lai_val
            })

df_900m = pd.DataFrame(data_900m)

print(f"\nTotal 900m grid cells with valid data: {len(df_900m)}")
print("\nFirst few rows:")
print(df_900m.head(10))

# Summary statistics
print(f"\nBurn severity weighted range: [{df_900m['burn_severity_weighted'].min():.3f}, {df_900m['burn_severity_weighted'].max():.3f}]")
print(f"LAI difference range: [{df_900m['lai_difference'].min():.3f}, {df_900m['lai_difference'].max():.3f}]")

# Visualize the 900m grids
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Weighted burn severity at 900m
ax1 = axes[0]
im1 = ax1.imshow(burn_severity_900m, cmap='YlOrRd', vmin=0, vmax=3)
ax1.set_title('Weighted Burn Severity\n(900m resolution)', fontsize=14, fontweight='bold')
ax1.axis('off')
cbar1 = plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
cbar1.set_label('Weighted Severity', fontsize=11)

# Panel 2: LAI difference at 900m
ax2 = axes[1]
im2 = ax2.imshow(lai_diff_900m, cmap='coolwarm', vmin=-2.5, vmax=2.5)
ax2.set_title('LAI Difference\n(900m resolution)', fontsize=14, fontweight='bold')
ax2.axis('off')
cbar2 = plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
cbar2.set_label('LAI Difference [-]', fontsize=11)

# Panel 3: Scatter plot of burn severity vs LAI difference
ax3 = axes[2]
scatter = ax3.scatter(df_900m['burn_severity_weighted'], df_900m['lai_difference'], 
                     c=df_900m['burn_severity_weighted'], cmap='YlOrRd', 
                     alpha=0.5, s=20, edgecolors='black', linewidth=0.5)
ax3.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax3.set_xlabel('Weighted Burn Severity', fontsize=12)
ax3.set_ylabel('LAI Difference (Pre-fire - Post-fire)', fontsize=12)
ax3.set_title('Burn Severity vs LAI Difference\n(900m grid cells)', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)
cbar3 = plt.colorbar(scatter, ax=ax3, fraction=0.046, pad=0.04)
cbar3.set_label('Weighted Severity', fontsize=11)

plt.tight_layout()
plt.show()

# # Save the combined 900m dataframe
# output_csv = os.path.join(selected_path, "burn_severity_lai_900m.csv")
# df_900m.to_csv(output_csv, index=False)
# print(f"\nSaved 900m data to: {output_csv}")

In [ ]:
## site selection based on 900m burn severity raster

In [ ]:
# Site selection strategy for 900m weighted burn severity grid
# 
# Strategy: Divide the weighted severity into ranges that correspond to the original categories,
# then select sites based on LAI percentiles within each range

import pandas as pd
import numpy as np

# Define severity ranges for the weighted burn severity
# Use quantiles of weighted burn severity for more data-driven ranges
# This ensures each category has sufficient samples
valid_severity = df_900m[df_900m['burn_severity_weighted'] >= 0]['burn_severity_weighted']
q25, q50, q75 = valid_severity.quantile([0.25, 0.5, 0.75])

print("Weighted burn severity quantiles (excluding negative values):")
print(f"  25th percentile: {q25:.3f}")
print(f"  50th percentile (median): {q50:.3f}")
print(f"  75th percentile: {q75:.3f}")
print(f"  Max: {valid_severity.max():.3f}")
print(f"  Min: {valid_severity.min():.3f}")

print("\n" + "="*70)
print("Using quantile-based severity ranges with percentile LAI selection")
print("="*70)

# Define ranges based on quantiles
quantile_ranges = {
    'increased_greenness': (df_900m['burn_severity_weighted'].min(), 0.0),
    'unburned_to_low': (valid_severity.min(), q25),
    'low_severity': (q25, q50),
    'moderate_severity': (q50, q75),
    'high_severity': (q75, valid_severity.max() + 0.01)
}

# Define LAI percentiles for each category
lai_percentiles = {
    'increased_greenness': 0.10,  # 10th percentile (most negative LAI difference)
    'unburned_to_low': 0.25,      # 25th percentile (lower LAI change)
    'low_severity': 0.50,         # 50th percentile (median LAI change)
    'moderate_severity': 0.75,    # 75th percentile (higher LAI change)
    'high_severity': 0.90         # 90th percentile (highest LAI change)
}

selected_sites_v2 = {}

for category, (min_sev, max_sev) in quantile_ranges.items():
    subset = df_900m[
        (df_900m['burn_severity_weighted'] >= min_sev) & 
        (df_900m['burn_severity_weighted'] < max_sev)
    ].copy()
    
    if len(subset) == 0:
        print(f"\n{category}: No data in range [{min_sev:.2f}, {max_sev:.2f})")
        continue
    
    print(f"\n{category.replace('_', ' ').title()}:")
    print(f"  Severity range: [{min_sev:.2f}, {max_sev:.2f})")
    print(f"  Number of cells: {len(subset)}")
    print(f"  LAI difference range: [{subset['lai_difference'].min():.3f}, {subset['lai_difference'].max():.3f}]")
    
    # Use percentile-based selection for all categories
    percentile = lai_percentiles[category]
    target_lai = subset['lai_difference'].quantile(percentile)
    label = f"{int(percentile*100)}th percentile"
    
    # Find the cell closest to the target LAI
    subset['lai_diff_from_target'] = np.abs(subset['lai_difference'] - target_lai)
    selected_site = subset.loc[subset['lai_diff_from_target'].idxmin()]
    
    selected_sites_v2[category] = selected_site
    
    print(f"  Target LAI ({label}): {target_lai:.3f}")
    print(f"  Selected site:")
    print(f"    Row, Col: ({int(selected_site['row'])}, {int(selected_site['col'])})")
    print(f"    Coordinates: ({selected_site['x_geo']:.2f}, {selected_site['y_geo']:.2f})")
    print(f"    Weighted severity: {selected_site['burn_severity_weighted']:.3f}")
    print(f"    LAI difference: {selected_site['lai_difference']:.3f}")

# Visualize selected sites
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# Plot weighted burn severity as background
im = ax.imshow(burn_severity_900m, cmap='YlOrRd', vmin=0, vmax=3, alpha=0.7)

# Use the new selected sites
selected_sites = selected_sites_v2

# Plot selected sites
colors_dict = {
    'increased_greenness': 'lime',
    'unburned_to_low': 'green',
    'low_severity': 'cyan',
    'moderate_severity': 'yellow',
    'high_severity': 'red'
}

for category, site in selected_sites.items():
    ax.plot(site['col'], site['row'], 'o', 
            color=colors_dict.get(category, 'white'),
            markersize=15, markeredgecolor='black', markeredgewidth=2,
            label=f"{category.replace('_', ' ').title()}")
    ax.text(site['col'], site['row'] - 1, f"S{list(selected_sites.keys()).index(category)+1}",
            ha='center', va='bottom', fontsize=10, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

ax.set_title('Selected Sites on 900m Weighted Burn Severity Map', 
             fontsize=16, fontweight='bold')
ax.legend(loc='upper right', fontsize=11, framealpha=0.9)
ax.axis('off')

cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Weighted Burn Severity', fontsize=12)

plt.tight_layout()
plt.show()

# Print summary of selection strategy
print("\n" + "="*70)
print("SELECTION STRATEGY SUMMARY")
print("="*70)
print("LAI Difference Percentiles Used:")
for category, percentile in lai_percentiles.items():
    print(f"  {category.replace('_', ' ').title()}: {int(percentile*100)}th percentile")

In [ ]:
# Create individual scatter plot with severity category boundaries marked
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Plot all data points
scatter = ax.scatter(df_900m['burn_severity_weighted'], 
                     df_900m['lai_difference'], 
                     c=df_900m['burn_severity_weighted'], 
                     cmap='YlOrRd', 
                     alpha=0.5, 
                     s=30, 
                     edgecolors='black', 
                     linewidth=0.5,
                     vmin=df_900m['burn_severity_weighted'].min(),  # Include negative values
                     vmax=3)

# Add horizontal line at y=0
ax.axhline(y=0, color='gray', linestyle='--', linewidth=1.5, alpha=0.5, zorder=1)

# Add vertical dashed lines for severity category boundaries
# Using the quantile-based ranges (positive values only)
severity_boundaries = {
    'Q1 (25th)': q25,
    'Q2 (50th)': q50,
    'Q3 (75th)': q75
}

colors_boundaries = {
    'Q1 (25th)': 'green',
    'Q2 (50th)': 'cyan',
    'Q3 (75th)': 'orange'
}

for label, value in severity_boundaries.items():
    ax.axvline(x=value, 
               color=colors_boundaries[label], 
               linestyle='--', 
               linewidth=2, 
               alpha=0.7,
               label=f'{label} = {value:.3f}',
               zorder=2)

# Add vertical line at x=0 to separate negative (increased greenness) from positive severity
ax.axvline(x=0, 
           color='darkgreen', 
           linestyle='--', 
           linewidth=2.5, 
           alpha=0.8,
           label='Zero boundary',
           zorder=2)

# Add text labels for severity categories
y_text_position = ax.get_ylim()[1] * 0.95  # Near top of plot

# Category positions (midpoints) - including increased greenness
cat_positions = {
    'Increased\nGreenness': (df_900m['burn_severity_weighted'].min() + 0) / 2,
    'Unburned\nto Low': (valid_severity.min() + q25) / 2,
    'Low\nSeverity': (q25 + q50) / 2,
    'Moderate\nSeverity': (q50 + q75) / 2,
    'High\nSeverity': (q75 + valid_severity.max()) / 2
}

cat_colors = {
    'Increased\nGreenness': 'lightgreen',
    'Unburned\nto Low': 'lightblue',
    'Low\nSeverity': 'lightcyan',
    'Moderate\nSeverity': 'lightyellow',
    'High\nSeverity': 'lightcoral'
}

for cat_name, x_pos in cat_positions.items():
    ax.text(x_pos, y_text_position, cat_name,
            ha='center', va='top', fontsize=10, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.5', 
                     facecolor=cat_colors.get(cat_name, 'white'), 
                     alpha=0.8, 
                     edgecolor='black'))

# Labels and title
ax.set_xlabel('Weighted Burn Severity', fontsize=13, fontweight='bold')
ax.set_ylabel('LAI Difference (Pre-fire - Post-fire)', fontsize=13, fontweight='bold')
ax.set_title('Burn Severity vs LAI Difference at 900m Resolution\nwith Severity Category Boundaries', 
             fontsize=15, fontweight='bold', pad=20)

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Weighted Burn Severity', fontsize=12, fontweight='bold')

# Add legend for boundary lines
ax.legend(loc='lower right', fontsize=10, framealpha=0.95)

# Optional: Add selected sites to the plot
for category, site in selected_sites.items():
    ax.plot(site['burn_severity_weighted'], 
            site['lai_difference'], 
            'o', 
            color=colors_dict.get(category, 'white'),
            markersize=12, 
            markeredgecolor='black', 
            markeredgewidth=2,
            zorder=10)
    ax.text(site['burn_severity_weighted'], 
            site['lai_difference'] + 0.1, 
            f"S{list(selected_sites.keys()).index(category)+1}",
            ha='center', va='bottom', fontsize=9, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9))

# Grid
ax.grid(True, alpha=0.3, zorder=0)

# Set x-axis limits to include ALL data (negative and positive)
x_range = df_900m['burn_severity_weighted'].max() - df_900m['burn_severity_weighted'].min()
ax.set_xlim(df_900m['burn_severity_weighted'].min() - 0.05*x_range, 
            df_900m['burn_severity_weighted'].max() + 0.05*x_range)

plt.tight_layout()
plt.show()

# Print info about increased greenness category
increased_greenness_data = df_900m[df_900m['burn_severity_weighted'] < 0]
print(f"\nIncreased Greenness category:")
print(f"  Number of cells: {len(increased_greenness_data)}")
print(f"  Severity range: [{increased_greenness_data['burn_severity_weighted'].min():.3f}, {increased_greenness_data['burn_severity_weighted'].max():.3f}]")
print(f"  LAI difference range: [{increased_greenness_data['lai_difference'].min():.3f}, {increased_greenness_data['lai_difference'].max():.3f}]")

## find the lat lon of hillslope end points

In [ ]:
# Zoom in to each selected 900m grid cell and plot detailed views
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import rasterio
from rasterio.windows import Window

# Buffer size around each 900m cell (in 900m units)
buffer = 2  # This means we'll show 2 additional 900m cells on each side (total ~4.5km x 4.5km)

# For each selected site
for site_idx, (category, site) in enumerate(selected_sites.items(), 1):
    row, col = int(site['row']), int(site['col'])
    
    print(f"\nProcessing Site {site_idx}: {category.replace('_', ' ').title()}")
    print(f"  Grid position: ({row}, {col})")
    print(f"  Center coordinates: ({site['x_geo']:.2f}, {site['y_geo']:.2f})")
    
    # Define the extent in 900m grid coordinates
    row_min = max(0, row - buffer)
    row_max = min(burn_severity_900m.shape[0], row + buffer + 1)
    col_min = max(0, col - buffer)
    col_max = min(burn_severity_900m.shape[1], col + buffer + 1)
    
    # Calculate corresponding extent in 30m burn severity grid
    # Each 900m pixel = 30 pixels of 30m
    aggregation_factor = 30
    burn_row_min = row_min * aggregation_factor
    burn_row_max = row_max * aggregation_factor
    burn_col_min = col_min * aggregation_factor
    burn_col_max = col_max * aggregation_factor
    
    # Extract the zoomed regions
    burn_30m_zoom = burn_severity_30m[burn_row_min:burn_row_max, burn_col_min:burn_col_max]
    
    # Calculate geographic extent for this window
    with rasterio.open(rdnbr_tif_fp) as src:
        # Get bounds of the zoomed window
        window = Window(burn_col_min, burn_row_min, 
                       burn_col_max - burn_col_min, 
                       burn_row_max - burn_row_min)
        zoom_bounds = rasterio.windows.bounds(window, src.transform)
        zoom_transform = rasterio.windows.transform(window, src.transform)
    
    # Extract LAI data for this extent
    # Convert zoom_bounds (in raster CRS) to lat/lon for LAI data
    from pyproj import Transformer
    transformer = Transformer.from_crs(raster_crs, 'EPSG:4326', always_xy=True)
    lon_min, lat_min = transformer.transform(zoom_bounds[0], zoom_bounds[1])
    lon_max, lat_max = transformer.transform(zoom_bounds[2], zoom_bounds[3])
    
    # Subset LAI data
    lai_zoom = LAI_data.sel(
        lon=slice(lon_min, lon_max),
        lat=slice(lat_max, lat_min)  # Note: lat is reversed
    )
    
    lai_prefire_zoom = lai_zoom.isel(time=t_lai_prefire)
    lai_postfire_zoom = lai_zoom.isel(time=t_lai_postfire)
    lai_diff_zoom = lai_prefire_zoom - lai_postfire_zoom
    
    # Create figure with 2x3 subplots
    fig = plt.figure(figsize=(20, 12))
    gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)
    
    fig.suptitle(f'Site {site_idx}: {category.replace("_", " ").title()}\n' + 
                 f'Grid ({row}, {col}) | Coords: ({site["x_geo"]:.2f}, {site["y_geo"]:.2f}) | ' +
                 f'Burn Severity: {site["burn_severity_weighted"]:.3f} | LAI Diff: {site["lai_difference"]:.3f}',
                 fontsize=16, fontweight='bold', y=0.98)
    
    # Panel (1,1): Burn Severity 30m
    ax1 = fig.add_subplot(gs[0, 0])
    im1 = ax1.imshow(burn_30m_zoom, cmap=cmap, norm=norm, 
                     extent=[zoom_bounds[0], zoom_bounds[2], zoom_bounds[1], zoom_bounds[3]])
    
    # Draw rectangle around the selected 900m grid cell
    cell_width = 900  # meters
    rect_x = site['x_geo'] - cell_width/2
    rect_y = site['y_geo'] - cell_width/2
    rect = Rectangle((rect_x, rect_y), cell_width, cell_width,
                     linewidth=3, edgecolor='magenta', facecolor='none', linestyle='--')
    ax1.add_patch(rect)
    
    ax1.set_title('Burn Severity (30m resolution)', fontsize=13, fontweight='bold')
    ax1.set_xlabel('Easting [m]', fontsize=11)
    ax1.set_ylabel('Northing [m]', fontsize=11)
    tick_positions = [0, 1, 2, 3, 4, 5]
    cbar1 = plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
    cbar1.set_ticks(tick_positions)
    cbar1.ax.set_yticklabels(["Non-Proc", "Unburned-Low", "Low", "Moderate", "High", "Inc. Green"], fontsize=9)
    # Overlay rivers AFTER the imshow
    watershed_workflow.plot.rivers(rivers, raster_crs, ax=ax1, color='lightgray', linewidth=1.5)
    ax1.set_xlim(zoom_bounds[0], zoom_bounds[2])
    ax1.set_ylim(zoom_bounds[1], zoom_bounds[3])
    ax1.grid(True, alpha=0.3)
    
    # Panel (1,2): DEM with rivers
    ax3 = fig.add_subplot(gs[0, 2])
    try:
        # Plot full DEM with extent
        im2 = ax3.imshow(dem2, cmap='terrain', extent=extent, origin='upper')
        
        # Draw rectangle around the selected 900m grid cell
        rect2 = Rectangle((rect_x, rect_y), cell_width, cell_width,
                         linewidth=3, edgecolor='magenta', facecolor='none', linestyle='--')
        ax3.add_patch(rect2)
        
        # Overlay rivers
        watershed_workflow.plot.rivers(rivers, raster_crs, ax=ax3, color='blue', linewidth=1.5)
        
        ax3.set_title('Digital Elevation Model (DEM)', fontsize=13, fontweight='bold')
        ax3.set_xlabel('Easting [m]', fontsize=11)
        ax3.set_ylabel('Northing [m]', fontsize=11)
        
        # Zoom in to the same extent as burn severity
        ax3.set_xlim(zoom_bounds[0], zoom_bounds[2])
        ax3.set_ylim(zoom_bounds[1], zoom_bounds[3])
        
        cbar2 = plt.colorbar(im2, ax=ax3, fraction=0.046, pad=0.04)
        cbar2.set_label('Elevation [m]', fontsize=10)
        ax3.grid(True, alpha=0.3)
    except Exception as e:
        print(f"  Warning: Could not plot DEM data for this site: {e}")
        ax3.text(0.5, 0.5, 'DEM data not available', 
                ha='center', va='center', fontsize=12, transform=ax3.transAxes)
        ax3.axis('off')
    
    # Panel (1,2): Empty
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.axis('off')
    
    # Panel (2,1): LAI Pre-fire
    ax4 = fig.add_subplot(gs[1, 0])
    im4 = lai_prefire_zoom.plot(ax=ax4, levels=np.linspace(0, 5, 51), 
                                 cmap='Spectral_r', extend='max', add_colorbar=False)
    
    # Draw rectangle around selected cell (in lat/lon)
    # Convert cell bounds to lat/lon
    cell_bounds_geo = [
        rect_x, rect_y,
        rect_x + cell_width, rect_y + cell_width
    ]
    lon1, lat1 = transformer.transform(cell_bounds_geo[0], cell_bounds_geo[1])
    lon2, lat2 = transformer.transform(cell_bounds_geo[2], cell_bounds_geo[3])
    rect4 = Rectangle((lon1, lat1), lon2-lon1, lat2-lat1,
                      linewidth=3, edgecolor='magenta', facecolor='none', linestyle='--')
    ax4.add_patch(rect4)
    
    # Plot rivers (after the LAI plot)
    watershed_workflow.plot.rivers(rivers, 'EPSG:4326', ax=ax4, color='cyan', linewidth=1.5)
    
    ax4.set_title(f'LAI Pre-fire\n{LAI_data.time[t_lai_prefire].dt.strftime("%Y-%m-%d").item()}', 
                  fontsize=13, fontweight='bold')
    ax4.set_xlabel('Longitude', fontsize=11)
    ax4.set_ylabel('Latitude', fontsize=11)
    ax4.set_xlim(lon_min, lon_max)
    ax4.set_ylim(lat_min, lat_max)
    cbar4 = plt.colorbar(im4, ax=ax4, fraction=0.046, pad=0.04)
    cbar4.set_label('LAI [-]', fontsize=10)
    ax4.grid(True, alpha=0.3)
    ax4.set_aspect('equal')
    
    # Panel (2,2): LAI Post-fire
    ax5 = fig.add_subplot(gs[1, 1])
    im5 = lai_postfire_zoom.plot(ax=ax5, levels=np.linspace(0, 5, 51), 
                                  cmap='Spectral_r', extend='max', add_colorbar=False)
    
    rect5 = Rectangle((lon1, lat1), lon2-lon1, lat2-lat1,
                      linewidth=3, edgecolor='magenta', facecolor='none', linestyle='--')
    ax5.add_patch(rect5)
    
    # Plot rivers (after the LAI plot)
    watershed_workflow.plot.rivers(rivers, 'EPSG:4326', ax=ax5, color='cyan', linewidth=1.5)
    
    ax5.set_title(f'LAI Post-fire\n{LAI_data.time[t_lai_postfire].dt.strftime("%Y-%m-%d").item()}', 
                  fontsize=13, fontweight='bold')
    ax5.set_xlabel('Longitude', fontsize=11)
    ax5.set_ylabel('Latitude', fontsize=11)
    ax5.set_xlim(lon_min, lon_max)
    ax5.set_ylim(lat_min, lat_max)
    cbar5 = plt.colorbar(im5, ax=ax5, fraction=0.046, pad=0.04)
    cbar5.set_label('LAI [-]', fontsize=10)
    ax5.grid(True, alpha=0.3)
    ax5.set_aspect('equal')
    
    # Panel (2,3): LAI Difference
    ax6 = fig.add_subplot(gs[1, 2])
    im6 = lai_diff_zoom.plot(ax=ax6, levels=np.linspace(-2.5, 2.5, 51), 
                             cmap='coolwarm', extend='both', add_colorbar=False)
    
    rect6 = Rectangle((lon1, lat1), lon2-lon1, lat2-lat1,
                      linewidth=3, edgecolor='magenta', facecolor='none', linestyle='--')
    ax6.add_patch(rect6)
    
    # Plot rivers (after the LAI plot)
    watershed_workflow.plot.rivers(rivers, 'EPSG:4326', ax=ax6, color='blue', linewidth=1.5)
    
    ax6.set_title('LAI Difference\n(Pre-fire - Post-fire)', fontsize=13, fontweight='bold')
    ax6.set_xlabel('Longitude', fontsize=11)
    ax6.set_ylabel('Latitude', fontsize=11)
    ax6.set_xlim(lon_min, lon_max)
    ax6.set_ylim(lat_min, lat_max)
    cbar6 = plt.colorbar(im6, ax=ax6, fraction=0.046, pad=0.04)
    cbar6.set_label('LAI Difference [-]', fontsize=10)
    ax6.grid(True, alpha=0.3)
    ax6.set_aspect('equal')
    
    # Save figure
    output_fig_path = os.path.join(site_selections_folder, 
                                   f"site{site_idx}_{category}_zoom_detail.png")
    plt.savefig(output_fig_path, dpi=150, bbox_inches='tight')
    print(f"  Saved figure to: {output_fig_path}")
    
    plt.show()

print("\n" + "="*70)
print("All site detail plots completed!")
print("="*70)

In [ ]:
# Identify river-boundary intersections with 900m grid cell for all selected sites
from shapely.geometry import box, LineString, Point
from shapely.ops import unary_union
import numpy as np

# Buffer size around each 900m cell (in 900m units)
buffer = 2  # Same as used in the zoom plots

# Store intersection points for each site
site_intersections = {}

# For each selected site
for site_idx, (category, site) in enumerate(selected_sites.items(), 1):
    row, col = int(site['row']), int(site['col'])
    
    print(f"\nProcessing Site {site_idx}: {category.replace('_', ' ').title()}")
    print(f"  Grid position: ({row}, {col})")
    print(f"  Center coordinates: ({site['x_geo']:.2f}, {site['y_geo']:.2f})")
    
    # Define the extent in 900m grid coordinates
    row_min = max(0, row - buffer)
    row_max = min(burn_severity_900m.shape[0], row + buffer + 1)
    col_min = max(0, col - buffer)
    col_max = min(burn_severity_900m.shape[1], col + buffer + 1)
    
    # Calculate corresponding extent in 30m burn severity grid
    # Each 900m pixel = 30 pixels of 30m
    aggregation_factor = 30
    burn_row_min = row_min * aggregation_factor
    burn_row_max = row_max * aggregation_factor
    burn_col_min = col_min * aggregation_factor
    burn_col_max = col_max * aggregation_factor
    
    # Calculate geographic extent for this window (for plotting)
    with rasterio.open(rdnbr_tif_fp) as src:
        # Get bounds of the zoomed window
        window = Window(burn_col_min, burn_row_min, 
                       burn_col_max - burn_col_min, 
                       burn_row_max - burn_row_min)
        zoom_bounds = rasterio.windows.bounds(window, src.transform)
    
    # Define the 900m grid cell rectangle
    cell_width = 900  # meters
    rect_x = site['x_geo'] - cell_width/2
    rect_y = site['y_geo'] - cell_width/2
    
    # Create a box for the 900m grid cell
    cell_box = box(rect_x, rect_y, rect_x + cell_width, rect_y + cell_width)
    
    # Find river-cell boundary intersections
    intersections = []
    
    for river in rivers:
        # Get all coordinates from the river tree
        for node in river.preOrder():
            river_line = node.segment
            if river_line.intersects(cell_box.boundary):
                intersection = river_line.intersection(cell_box.boundary)
                if intersection.geom_type == 'Point':
                    intersections.append((intersection.x, intersection.y))
                elif intersection.geom_type == 'MultiPoint':
                    for pt in intersection.geoms:
                        intersections.append((pt.x, pt.y))
                elif intersection.geom_type == 'LineString':
                    # If intersection is a line, get endpoints
                    coords = list(intersection.coords)
                    intersections.append(coords[0])
                    intersections.append(coords[-1])
                elif intersection.geom_type == 'GeometryCollection' or intersection.geom_type == 'MultiLineString':
                    for geom in intersection.geoms:
                        if geom.geom_type == 'Point':
                            intersections.append((geom.x, geom.y))
    
    # Remove duplicates (points very close to each other)
    unique_intersections = []
    tolerance = 10  # meters (reduced tolerance for 900m cell)
    for pt in intersections:
        is_duplicate = False
        for upt in unique_intersections:
            if np.sqrt((pt[0]-upt[0])**2 + (pt[1]-upt[1])**2) < tolerance:
                is_duplicate = True
                break
        if not is_duplicate:
            unique_intersections.append(pt)
    
    # Store results
    site_intersections[category] = {
        'site_info': site,
        'zoom_bounds': zoom_bounds,
        'cell_bounds': (rect_x, rect_y, rect_x + cell_width, rect_y + cell_width),
        'intersections': unique_intersections
    }
    
    print(f"  Found {len(unique_intersections)} river-cell boundary intersection points")
    print(f"  Cell bounds: [{rect_x:.2f}, {rect_y:.2f}, {rect_x + cell_width:.2f}, {rect_y + cell_width:.2f}]")
    print(f"  Intersection coordinates:")
    for idx, int_pt in enumerate(unique_intersections, 1):
        print(f"    P{idx}: ({int_pt[0]:.2f}, {int_pt[1]:.2f})")

print("\n" + "="*70)
print("River-cell boundary intersection identification completed!")
print(f"Total sites processed: {len(site_intersections)}")
print("="*70)

In [ ]:
# Plot DEM and river networks with intersection points for each site
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

for site_idx, (category, data) in enumerate(site_intersections.items(), 1):
    site = data['site_info']
    zoom_bounds = data['zoom_bounds']
    cell_bounds = data['cell_bounds']
    intersections = data['intersections']
    
    print(f"\nPlotting Site {site_idx}: {category.replace('_', ' ').title()}")
    print(f"  Number of intersection points: {len(intersections)}")
    
    # Create figure
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    
    # Plot DEM
    im = ax.imshow(dem2, cmap='terrain', extent=extent, origin='upper')
    
    # Draw rectangle around the selected 900m grid cell
    cell_width = 900  # meters
    rect_x = cell_bounds[0]
    rect_y = cell_bounds[1]
    rect = Rectangle((rect_x, rect_y), cell_width, cell_width,
                     linewidth=3, edgecolor='magenta', facecolor='none', linestyle='--',
                     label='Selected 900m grid cell')
    ax.add_patch(rect)
    
    # Overlay rivers
    watershed_workflow.plot.rivers(rivers, raster_crs, ax=ax, color='blue', linewidth=2)
    
    # Plot intersection points with labels
    for idx, int_pt in enumerate(intersections, 1):
        ax.plot(int_pt[0], int_pt[1], 'ro', markersize=10, 
                markeredgecolor='white', markeredgewidth=2, zorder=10,
                label='River-cell intersection' if idx == 1 else '')
        ax.text(int_pt[0], int_pt[1], f'  P{idx}', 
                fontsize=10, fontweight='bold', color='red',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='red'),
                zorder=11)
    
    # Plot site center
    ax.plot(site['x_geo'], site['y_geo'], 'k*', markersize=20, 
            markeredgecolor='white', markeredgewidth=1.5, zorder=10,
            label='Site center')
    
    # Set limits to zoom to the region
    ax.set_xlim(zoom_bounds[0], zoom_bounds[2])
    ax.set_ylim(zoom_bounds[1], zoom_bounds[3])
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Elevation [m]', fontsize=12, fontweight='bold')
    
    # Labels and title
    ax.set_xlabel('Easting [m]', fontsize=12, fontweight='bold')
    ax.set_ylabel('Northing [m]', fontsize=12, fontweight='bold')
    ax.set_title(f'Site {site_idx}: {category.replace("_", " ").title()}\n' +
                 f'DEM with River Networks and 900m Cell Boundary Intersections\n' +
                 f'Burn Severity: {site["burn_severity_weighted"]:.3f} | LAI Diff: {site["lai_difference"]:.3f}',
                 fontsize=14, fontweight='bold', pad=20)
    
    # Add legend
    ax.legend(loc='upper right', fontsize=11, framealpha=0.95)
    
    # Add grid
    ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
    
    # Save figure
    output_fig_path = os.path.join(site_selections_folder, 
                                   f"site{site_idx}_{category}_DEM_rivers_intersections.png")
    plt.savefig(output_fig_path, dpi=150, bbox_inches='tight')
    print(f"  Saved figure to: {output_fig_path}")
    
    plt.show()

print("\n" + "="*70)
print("All DEM-river intersection plots completed!")
print("="*70)

In [ ]:
# Manually select intersection points for each site
from pyproj import Transformer

# Create transformer from raster CRS to lat/lon
transformer = Transformer.from_crs(raster_crs, 'EPSG:4326', always_xy=True)

# Define manual point selections for each site (1-indexed as shown in plots)
manual_selections = {
    'increased_greenness': [1, 3, 5, 6],  # Site 1
    'unburned_to_low': [],                 # Site 2
    'low_severity': [1, 2],                # Site 3
    'moderate_severity': [1, 2, 4, 5, 6, 7],  # Site 4
    'high_severity': [1, 2, 3, 4, 5, 6, 7]    # Site 5
}

# Store selected points with their coordinates
selected_points = {}

print("="*70)
print("MANUALLY SELECTED INTERSECTION POINTS")
print("="*70)

for site_idx, (category, data) in enumerate(site_intersections.items(), 1):
    site = data['site_info']
    intersections = data['intersections']
    selected_indices = manual_selections.get(category, [])
    
    print(f"\nSite {site_idx}: {category.replace('_', ' ').title()}")
    print(f"  Total intersection points available: {len(intersections)}")
    print(f"  Selected points: {selected_indices if selected_indices else 'None'}")
    
    # Store selected points with coordinates
    site_selected_points = []
    
    if selected_indices:
        print(f"\n  Selected point coordinates:")
        print(f"  {'Point':<8} {'Easting [m]':<15} {'Northing [m]':<15} {'Longitude':<15} {'Latitude':<15}")
        print(f"  {'-'*8} {'-'*15} {'-'*15} {'-'*15} {'-'*15}")
        
        for idx in selected_indices:
            if 1 <= idx <= len(intersections):
                # Get coordinates (converting from 1-indexed to 0-indexed)
                easting, northing = intersections[idx - 1]
                
                # Convert to lat/lon
                lon, lat = transformer.transform(easting, northing)
                
                site_selected_points.append({
                    'point_id': idx,
                    'easting': easting,
                    'northing': northing,
                    'longitude': lon,
                    'latitude': lat
                })
                
                print(f"  P{idx:<7} {easting:<15.2f} {northing:<15.2f} {lon:<15.6f} {lat:<15.6f}")
            else:
                print(f"  Warning: P{idx} is out of range (only {len(intersections)} points available)")
    else:
        print(f"  No points selected for this site")
    
    selected_points[category] = {
        'site_idx': site_idx,
        'site_info': site,
        'selected_points': site_selected_points,
        'num_selected': len(site_selected_points)
    }

print("\n" + "="*70)
print("SUMMARY")
print("="*70)
for site_idx, (category, data) in enumerate(selected_points.items(), 1):
    print(f"Site {site_idx} ({category.replace('_', ' ').title()}): {data['num_selected']} points selected")

print("\n" + "="*70)

# Optionally, save to a CSV file for each site
import pandas as pd

# for category, data in selected_points.items():
#     if data['num_selected'] > 0:
#         site_idx = data['site_idx']
#         df = pd.DataFrame(data['selected_points'])
        
#         # Add site information
#         df['site_name'] = category
#         df['site_index'] = site_idx
        
#         # Reorder columns
#         df = df[['site_name', 'site_index', 'point_id', 'easting', 'northing', 'longitude', 'latitude']]
        
#         # Save to CSV
#         output_csv = os.path.join(site_selections_folder, 
#                                   f"site{site_idx}_{category}_selected_intersection_points.csv")
#         df.to_csv(output_csv, index=False)
#         print(f"Saved: {output_csv}")

# Create a combined CSV with all selected points
all_points = []
for category, data in selected_points.items():
    for pt in data['selected_points']:
        pt_copy = pt.copy()
        pt_copy['site_name'] = category
        pt_copy['site_index'] = data['site_idx']
        all_points.append(pt_copy)

if all_points:
    df_all = pd.DataFrame(all_points)
    df_all = df_all[['site_name', 'site_index', 'point_id', 'easting', 'northing', 'longitude', 'latitude']]
    output_csv_all = os.path.join(site_selections_folder, "all_selected_intersection_points.csv")
    df_all.to_csv(output_csv_all, index=False)
    print(f"\nSaved combined file: {output_csv_all}")
    print(f"Total selected points across all sites: {len(all_points)}")